# AutoGluon.TimeSeries V1.4 Cheatsheet

This notebook contains the key code snippets from the AutoGluon Time Series cheatsheet.

## Installation

AutoGluon (GitHub) supports Python 3.9 to 3.12 and is available for Linux, macOS, and Windows. The fastest way to install AutoGluon is through the `uv` package manager.

```bash
# Install uv package manager (faster than pip)
# !python -m pip install -U uv

# Install AutoGluon
# !uv pip install autogluon

# Or install with pip
# !python -m pip install autogluon
```

## Preparing Data

AutoGluon can generate forecasts for datasets consisting of **multiple univariates** time series. Here we use the M4 Competition Daily dataset to demonstrate how to do forecasting with AutoGluon. 

The data typically requires two parts: **raw data** (time series) and **static features** (metadata).

In [ ]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# 1. Load the raw time series data (time series values)
# NOTE: You would replace 'm4_daily.csv' with your own dataset path.
raw_data = pd.read_csv("m4_daily.csv")
print("Raw Data Head:")
print(raw_data.head())

# 2. Load the static features (metadata for each time series ID)
static_features = pd.read_csv("m4_metadata.csv")
print("\nStatic Features Head:")
print(static_features.head())

### Convert Raw Data into a TimeSeriesDataFrame

Convert your raw data into the format required by AutoGluon: `TimeSeriesDataFrame`.

In [ ]:
# from autogluon.timeseries import TimeSeriesDataFrame # Already imported above
train_data = TimeSeriesDataFrame(
    raw_data,
    id_column="item_id",
    timestamp_column="timestamp",
    static_features=static_features,  # Optional metadata/covariates
)

print("Processed Training Data:")
print(train_data.head())

## Training

Train models to forecast the values in the column `target` $30$ steps into the future.

In [ ]:
# from autogluon.timeseries import TimeSeriesPredictor # Already imported above

predictor = TimeSeriesPredictor(
    target="target",
    prediction_length=30, # The number of time steps into the future to forecast
    # Optional: Additional covariates that are known in the future
    # known_covariates_names=['weekday', 'month'],
).fit(
    train_data,
    # Presets control the model quality and training time
    presets="medium_quality",
    # Other options: tuning metric, time limit, hyperparamter adjustment, etc.
    # eval_metric="MAPE",
    # time_limit=600,
    # verbosity=2
)

## Predicting

Forecast `prediction_length` steps into the future starting from the end of each time series in `train_data`.

In [ ]:
predictions = predictor.predict(
    train_data,
    # If you used known_covariates_names during training, pass them here:
    # known_covariates=known_covariates,
)

print("Forecasted Predictions Head:")
print(predictions.head())

# AutoGluon generates probabilistic forecasts that include:
# - mean_forecast - expected value of the time series
# - quantile_forecast - range of possible outcomes

# Predict on a new, unseen dataset
# predictions_test = predictor.predict_test_data(
#     test_data,
#     model_names=['DeepAR', 'ETS'], # Optional: specify which models to use
# )

## Model Understanding

Understand the contribution of each model.

In [ ]:
leaderboard = predictor.leaderboard()
print(leaderboard)